# 01 — R Fundamentals

**R syntax, data types, data frames, and base plotting using stem cell gene expression data**

| | |
|---|---|
| **Duration** | ~2 hours |
| **Prerequisites** | None — this is the starting point for beginners |
| **Packages** | `base R (no packages needed)` |

## Learning Objectives

- Understand R syntax: assignment, comments, function calls, and getting help
- Work with R data types and structures: vectors, lists, data frames, and factors
- Index and subset data using [], $, and logical conditions
- Create basic plots with base R graphics
- Read and write CSV files

---

*Part of the R for Bioinformatics Applications workshop series — Stanford Institute for Stem Cell Biology and Regenerative Medicine*



---

## Setup


This notebook uses **base R only** — no package installation needed.

We will work with data from a **human embryonic stem cell (ESC) differentiation study** (Koh et al., 2016). The data files should be in the `data/` folder:

- `stemcell_cell_metadata.csv` — metadata for 531 cells across 9 differentiation stages
- `stemcell_expression_top50.csv` — expression counts for the 50 most variable genes across those cells

If you don't have the data folder, set `data_dir` below to the correct path.

In [ ]:
# Set the path to the data directory
data_dir <- if (dir.exists("data")) "data" else if (dir.exists("../data")) "../data" else "data"  # Change this if your data is elsewhere

# Verify the files exist
metadata_file <- file.path(data_dir, "stemcell_cell_metadata.csv")
expression_file <- file.path(data_dir, "stemcell_expression_top50.csv")

if (file.exists(metadata_file)) {
  cat("✓ Cell metadata file found\n")
} else {
  cat("✗ Cell metadata file NOT found at:", metadata_file, "\n")
}

if (file.exists(expression_file)) {
  cat("✓ Expression file found\n")
} else {
  cat("✗ Expression file NOT found at:", expression_file, "\n")
}



---

## Fast-Track Self-Check (Optional)


**Already comfortable with R?** Try these 5 quick exercises. If you can complete them all, skip to **Notebook 02: Data Wrangling with the Tidyverse**.

If you get stuck on any of these, work through the rest of this notebook — it covers everything you need.

**FT-1.** Read `stemcell_cell_metadata.csv` into a data frame called `cell_meta`. How many rows and columns does it have?

**FT-2.** What are the unique values in the `phenotype` column?

**FT-3.** Filter the data frame to only include cells where `total_counts` is greater than 1,000,000. How many cells pass this filter?

**FT-4.** Create a boxplot of `total_features` grouped by `phenotype`.

**FT-5.** Calculate the mean and median of `total_counts` for each phenotype using `tapply()`.

> Solutions are in the companion solutions notebook.


---

## 1. R Syntax Basics


### Assignment

In R, we assign values to variables using `<-` (the preferred R style) or `=`. Both work, but `<-` is the R convention.

```r
gene_name <- "POU5F1"    # character (text)
fold_change <- 2.5       # numeric
is_significant <- TRUE   # logical (TRUE/FALSE)
```

### Comments

Anything after `#` is a comment and is ignored by R:

```r
# This is a comment
gene_count <- 500  # This is also a comment
```

### Function Calls

R uses functions extensively. Functions take arguments inside parentheses:

```r
print("Hello, stem cells!")
sqrt(16)       # square root
length(c(1,2,3))  # length of a vector
```

### Getting Help

- `?function_name` opens help for a function
- `??search_term` searches all help pages
- `help("function_name")` is equivalent to `?`

In [ ]:
# Let's try some basic R operations
# Assignment
gene_name <- "POU5F1"       # OCT4, a key pluripotency factor
fold_change <- 2.5
is_significant <- TRUE
num_genes <- 500

# Print values
print(gene_name)
print(fold_change)
print(is_significant)

# Basic arithmetic
log2_fold_change <- log2(fold_change)  # log2 transform (common in genomics)
cat("log2 fold change:", log2_fold_change, "\n")

# Function calls
cat("Square root of 16:", sqrt(16), "\n")
cat("Number of genes:", num_genes, "\n")

# Getting help (uncomment to try)
# ?log
# ??heatmap



---

## 2. Data Types and Structures


### Basic Data Types

R has several core data types:

| Type | Example | Description |
|---|---|---|
| `numeric` | `2.5`, `100` | Numbers (decimals or integers) |
| `character` | `"POU5F1"` | Text strings |
| `logical` | `TRUE`, `FALSE` | Boolean values |
| `factor` | `factor(c("ESC","MPS"))` | Categorical data with defined levels |

### Vectors

A **vector** is the most common data structure in R — a sequence of elements of the **same type**. Create vectors with `c()` (combine):

```r
gene_names <- c("POU5F1", "NANOG", "SOX2", "KLF4")  # character vector
fold_changes <- c(1.5, 2.3, -0.8, 0.5)              # numeric vector
```

### Data Frames

A **data frame** is a table (like a spreadsheet) where each column can be a different type. This is the most common structure for bioinformatics data. We'll load one from CSV shortly.

### Inspecting Objects

- `str()` — structure of an object (most useful!)
- `class()` — the type/class of an object
- `summary()` — statistical summary
- `head()` — first few rows
- `dim()` — dimensions (rows × columns)
- `names()` or `colnames()` — column names

In [ ]:
# --- Vectors ---
# Create vectors of stem cell gene data
gene_names <- c("POU5F1", "NANOG", "SOX2", "KLF4", "MYC")
fold_changes <- c(2.5, 3.1, -0.8, 0.5, 1.2)
p_values <- c(0.001, 0.0001, 0.45, 0.03, 0.12)

# Inspect vectors
cat("Gene names:", gene_names, "\n")
cat("Fold changes:", fold_changes, "\n")
cat("Length of gene_names:", length(gene_names), "\n")
cat("Class of fold_changes:", class(fold_changes), "\n")

# Vector operations (element-wise)
log2_fc <- log2(fold_changes)
cat("log2 fold changes:", round(log2_fc, 3), "\n")

# Logical vector
significant <- p_values < 0.05
cat("Significant?", significant, "\n")


In [ ]:
# --- Factors ---
# Factors represent categorical data — important for experimental groups
phenotypes <- c("H7hESC", "H7hESC", "H7_derived_APS", "H7_derived_MPS", "H7_derived_APS")
phenotype_factor <- factor(phenotypes)

cat("Factor levels:", levels(phenotype_factor), "\n")
cat("Summary (counts per level):\n")
print(table(phenotype_factor))

# Factors are important for DESeq2 and other Bioconductor packages
# where experimental design is specified via factor levels


In [ ]:
# --- Lists ---
# Lists can hold different types of elements
gene_info <- list(
  name = "POU5F1",
  aliases = c("OCT4", "OCT3/4"),
  fold_change = 2.5,
  p_value = 0.001,
  is_marker = TRUE
)

# Access list elements with $ or [[ ]]
cat("Gene name:", gene_info$name, "\n")
cat("Aliases:", gene_info$aliases, "\n")
cat("Fold change:", gene_info[["fold_change"]], "\n")

# str() shows the structure of any object
str(gene_info)



---

## 3. Reading Data into R


Now let's load our stem cell data. The `read.csv()` function reads CSV files into data frames.

Our data comes from a study of **human H7 embryonic stem cells differentiating into mesoderm lineages** (Koh et al., 2016). The metadata file contains information about each cell, including which differentiation stage it was in.

In [ ]:
# Read the cell metadata CSV
cell_meta <- read.csv(metadata_file)

# Inspect the data frame
cat("=== Structure ===\n")
str(cell_meta)

cat("\n=== First 6 rows ===\n")
head(cell_meta)

cat("\n=== Dimensions ===\n")
cat("Rows:", nrow(cell_meta), " Columns:", ncol(cell_meta), "\n")

cat("\n=== Column names ===\n")
colnames(cell_meta)

cat("\n=== Summary ===\n")
summary(cell_meta)


In [ ]:
# Read the expression data (top 50 most variable genes)
expr_data <- read.csv(expression_file, row.names = 1)  # row.names=1 uses first column as row names

cat("=== Expression data structure ===\n")
cat("Dimensions:", nrow(expr_data), "genes x", ncol(expr_data), "cells\n")
cat("First 5 genes (row names):\n")
print(rownames(expr_data)[1:5])
cat("\nFirst 5 rows, first 5 columns:\n")
expr_data[1:5, 1:5]

cat("\n=== Summary of first few columns ===\n")
summary(expr_data[, 1:4])



---

## 4. Indexing and Subsetting


### Data Frame Indexing

Access data in data frames using several methods:

| Syntax | What it does |
|---|---|
| `df[i, j]` | Row i, column j |
| `df[i, ]` | Row i (all columns) |
| `df[, j]` | Column j (all rows) |
| `df$col` | A single column by name |
| `df[, "col"]` | Same as above, by name |
| `df[df$col > x, ]` | Rows where col > x (logical indexing) |

### Logical Indexing

This is one of the most powerful patterns in R — filter rows based on a condition:

```r
# Get cells with high total counts
high_count_cells <- cell_meta[cell_meta$total_counts > 2000000, ]
```

In [ ]:
# --- Basic indexing ---
# Single column with $
cat("First 5 phenotype values:\n")
print(head(cell_meta$phenotype, 5))

# Single column with [ ]
cat("\nFirst 5 total_counts values:\n")
print(head(cell_meta[, "total_counts"], 5))

# Specific rows and columns
cat("\nRows 1-3, columns 'cell_id' and 'phenotype':\n")
print(cell_meta[1:3, c("cell_id", "phenotype")])


In [ ]:
# --- Logical indexing (filtering) ---
# Filter cells with total_counts > 2,000,000
high_count <- cell_meta$total_counts > 2000000
cat("Cells with total_counts > 2M:", sum(high_count), "out of", nrow(cell_meta), "\n")

# Apply the filter
high_count_cells <- cell_meta[high_count, ]
head(high_count_cells)

# Multiple conditions: ESC cells with high feature counts
esc_cells <- cell_meta$phenotype == "H7hESC"
high_feature_esc <- cell_meta[esc_cells & cell_meta$total_features > 4000, ]
cat("\nESC cells with >4000 features:", nrow(high_feature_esc), "\n")
print(head(high_feature_esc))


In [ ]:
# --- The subset() function ---
# A more readable way to filter
esc_only <- subset(cell_meta, phenotype == "H7hESC")
cat("Number of ESC cells:", nrow(esc_only), "\n")

# Select specific columns too
esc_summary <- subset(cell_meta, phenotype == "H7hESC", 
                      select = c(cell_id, total_counts, total_features))
head(esc_summary)

# Unique values in a column
cat("\nAll unique phenotypes:\n")
print(unique(cell_meta$phenotype))

# Count cells per phenotype
cat("\nCells per phenotype:\n")
print(table(cell_meta$phenotype))


In [ ]:
# --- Indexing the expression matrix ---
# Get expression of a specific gene across all cells
# (Remember: genes are rows, cells are columns in expr_data)
gene_idx <- 1  # first gene
cat("Gene:", rownames(expr_data)[gene_idx], "\n")
cat("Expression in first 5 cells:\n")
print(expr_data[gene_idx, 1:5])

# Get expression of a specific cell across all genes
cell_idx <- 1
cat("\nExpression of first 10 genes in cell_1:\n")
print(expr_data[1:10, cell_idx])

# Get a subset: first 10 genes, first 5 cells
cat("\nSubmatrix (10 genes x 5 cells):\n")
print(expr_data[1:10, 1:5])



---

## 5. Basic Plotting with Base R


R has built-in plotting functions that are quick and useful for exploration:

- `plot()` — scatter plots
- `hist()` — histograms
- `boxplot()` — box plots (great for comparing groups)
- `barplot()` — bar charts

These are "base R" graphics. Later notebooks will introduce **ggplot2**, which produces more polished, publication-quality figures.

In [ ]:
# --- Histogram: distribution of total counts per cell ---
hist(cell_meta$total_counts,
     breaks = 30,
     col = "steelblue",
     main = "Distribution of Total Counts per Cell",
     xlab = "Total Counts",
     border = "white")

# Add a vertical line at the median
abline(v = median(cell_meta$total_counts), col = "red", lwd = 2, lty = 2)


In [ ]:
# --- Boxplot: total features by phenotype ---
boxplot(total_features ~ phenotype,
        data = cell_meta,
        las = 2,           # rotate x-axis labels
        col = "lightgreen",
        main = "Total Features per Cell by Differentiation Stage",
        xlab = "",
        ylab = "Total Features (detected genes)")


In [ ]:
# --- Scatter plot: total_counts vs total_features ---
plot(cell_meta$total_counts, cell_meta$total_features,
     pch = 19,             # solid circles
     col = rgb(0, 0, 1, 0.5),  # semi-transparent blue
     main = "Total Counts vs Total Features",
     xlab = "Total Counts",
     ylab = "Total Features")

# Add a trend line
abline(lm(total_features ~ total_counts, data = cell_meta), 
       col = "red", lwd = 2)


In [ ]:
# --- Barplot: cells per phenotype ---
pheno_counts <- table(cell_meta$phenotype)
barplot(pheno_counts,
        las = 2,
        col = "coral",
        main = "Number of Cells per Differentiation Stage",
        ylab = "Number of Cells",
        border = NA)



---

## 6. Writing Data


You can save data frames to CSV files with `write.csv()`:

```r
write.csv(my_data, "output_file.csv", row.names = FALSE)
```

Setting `row.names = FALSE` prevents R from adding an extra column with row numbers.

In [ ]:
# Example: save the high-count cells to a new CSV
high_count_cells <- subset(cell_meta, total_counts > 2000000)
cat("Saving", nrow(high_count_cells), "high-count cells to CSV\n")

# Write to a file (uncomment to actually write)
# write.csv(high_count_cells, "high_count_cells.csv", row.names = FALSE)

# For this exercise, let's just show what would be written
cat("\nFirst few rows of what would be saved:\n")
print(head(high_count_cells))



---

## Exercises

> **Instructions**: Complete the exercises below. Hints are provided in collapsible sections.  
> Check your answers against the companion **Solutions** notebook for this module.


### Exercise 1: Filter cells by total counts

Load `stemcell_cell_metadata.csv` and filter to only include cells where `total_counts` is between 500,000 and 3,000,000. How many cells pass this filter? What are the counts per phenotype in the filtered data?

<details>
<summary>💡 Hint</summary>

Use `subset()` with an `&` condition, then `table()` on the phenotype column.

</details>

### Exercise 2: Mean expression of a gene across phenotypes

Using the expression data (`expr_data`), pick any gene (row) and calculate the mean expression of that gene for cells belonging to each phenotype. 

*Hint: You'll need to match cell IDs between `expr_data` (column names) and `cell_meta` (cell_id column).*

<details>
<summary>💡 Hint</summary>

Use `colnames(expr_data)` to get cell IDs, match them to `cell_meta$cell_id`, then use `tapply()` with the phenotype as the grouping factor.

</details>

### Exercise 3: Boxplot of total_features by phenotype

Create a boxplot showing the distribution of `total_features` for each phenotype. Color the boxes differently for each phenotype. Add a horizontal line at the overall median.

<details>
<summary>💡 Hint</summary>

Use `boxplot()` with a formula. For colors, pass a vector of color names to the `col` argument. Use `abline(h = ...)` for the horizontal line.

</details>

### Exercise 4: Read expression CSV, compute row means, write top 10

Read `stemcell_expression_top50.csv`, compute the mean expression for each gene (across all cells), sort genes by mean expression (descending), and write the top 10 genes and their mean expression to a new CSV file called `top10_expressed_genes.csv`.

<details>
<summary>💡 Hint</summary>

Use `rowMeans()` on the expression matrix, `sort()` with `decreasing=TRUE`, then create a data frame with `data.frame()` and write with `write.csv()`.

</details>


---

## Wrap-Up

### Key Functions & Concepts

- `<- (assignment)`
- `c() (combine into vector)`
- `factor() (categorical data)`
- `read.csv() / write.csv()`
- `str() / class() / summary() / head()`
- `df[i,j] / df$col / subset() (indexing)`
- `plot() / hist() / boxplot() / barplot()`
- `table() (counting categories)`
- `abline() / lm() (adding trend lines)`

### Next Steps

Proceed to **Notebook 02: Data Wrangling with the Tidyverse**.

### Further Reading

- R for Data Science (free book): r4ds.had.co.nz
- Quick-R: statmethods.net
- R cheat sheets: rstudio.com/resources/cheatsheets/

---

*This notebook is part of the R for Bioinformatics Applications workshop series.*
